
   Experimento 6 - SINGLE CLASSIFIER Com uma rede apenas, sem patch clf + INPUT Resize

   Instancia modelos PURE single clf, usa imagem inteira (full resolution) CBIS
    e faz resize para entrar na rede. Resize pode ser Learning to resize
   
   Data Nov/21 ~ Jan/22


   Objetivo
   - Usar novo dataset 2304x1792 (Double Size)
   - Realizar Estrategia de resize (ex. Learn to Resize) para 1152x896
   - Passar nas diversas redes
   - Salvar resultado em arquivo para rodar sucessivas vezes
   - Ralizar diversos runs (3 ou 5) para ter média de execuções

   - (11-01-2021)
   - Segundo conclusoes do input size das redes pelo professor podemos
     reduzir a entrada 1152x896 pela metade para aproximar do tamanho do bloco
     das redes menores como EffcientNet-B0. Reduzir por 0.5 => 576x448, estará
     na ordem de 200%-300% e deve dar bons resultados na B0


  Data: Oct/2023

   - Objetivo: avaliar esse experimento com novas redes e GPU A100
   - Usar tempo collab pro+ pago.
   - Refatorar código - deixar com ultimas melhorias dos demais exps.
   

---
Check also single_clf_EXP6.py file on PC




## Single Image Classifiers on CBIS-DDSM - Original test set division

Data: Nov/21, Jan/22 / After paper - SR  / Update in Oct/2023

April/2024: run newer classifiers, include fixed resize.

May/2024: Include random init

Jun/2024: Bring VinDr support from EXP5C

V2: Support running in local server (home)

V3: Versao 2025/2026
- Including support to train in one dataset and fine tune in another (TRANSFER)
- Test model in another dataset (ONLY_TEST_DATASET) 29-03-2026
- Integrado Patch Based 31-03-2026
- "Top Layer" do Patch based otimizada, tamanho reduziu de 83 -> 36 MB no caso da rede pbc 
  EfficienNet-B0 e resultados ficaram iguais ou melhores para essa rede. Confirmar com outras.

TODO: Integrar nesse codigo a two-views.



In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

COLLAB = False

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
!nvidia-smi

Mon Jun 29 20:12:38 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.309.01             Driver Version: 535.309.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 2060 ...    Off | 00000000:01:00.0  On |                  N/A |
|  0%   46C    P5              22W / 184W |    444MiB /  8192MiB |     38%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
if COLLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

In [ ]:
# Defines - MEXER APENAS AQUI

DATASET = 'CBIS-DDSM'           #'CBIS-DDSM' VINDR_MAMMO VINDR_MAMMO-SMALL

NET_TYPE = 'SINGLE_PURE'        # 'SINGLE_PURE' 'PATCH_BASED' 'LEARN2RESIZE'  'FIXED_RESIZE'  'RANDOM_WEIGHTS' TRANSFER' 'ONLY_TEST' 

ONLY_TEST_DATASET = None        # None is the same or 'VINDR_MAMMO' or  'CBIS-DDSM'  # ex. treinar em CBIS testa em vindr- not fine tune.

INPUT_SIZE = '1152x896'         # '1152x896'  '2304x1792'


# For TRANSFER & ONLY_TEST_DATASET:
ORIGINAL_RESULT_FILE = './dc_single_pure_results_CBIS-DDSM_2026-03-30-10h53m/dc_single_pure_1152x896_SINGLE_PURE_test_result.json'

REUSE_PATH = False       # To append new models results in existent folder, if add more models in other day experiment

NUM_EPOCHS = 2  # 30: todas redes pretreinadas
# MINI_BATCH = Auto
NUM_WORKERS = 4
number_runs = 2 if NET_TYPE != 'ONLY_TEST' else 1
LR = 1e-4                       # for fixed LR


PRINT_LAYER_NAMES = False       # Print model layers for debug

# COLLAB/Gdrive DEFINES
if NET_TYPE == 'PATCH_BASED':
    project_name = 'pbc'        # "PBC" - Patch based classifier = patch + single classifiers
else:
    project_name = 'dc'         # dc: direct classifier - Single - apenas uma rede direta

project_name += '_'+NET_TYPE.lower()

# for folder timestamp
import time
import datetime
ts = time.time()
st = datetime.datetime.fromtimestamp(ts).strftime('%Y-%m-%d-%Hh%Mm')

if REUSE_PATH:
    save_path = './dc_single_pure_results_CBIS-DDSM_2026-04-05-18h10m/'
else:
    if COLLAB:
        collab_data_path = ''  # dataset is placed in current folder
        save_path = '/content/drive/My Drive/sinai/'+ project_name +'_results_'+DATASET+'_'+str(st)+'/'
    else:
        save_path = save_path = './'+ project_name +'_results_'+DATASET+'_'+str(st)+'/'

print(save_path)

./dc_single_pure_results_CBIS-DDSM_2026-06-29-20h12m/


In [6]:
# Models to run

# Low Memory / use bs=8

#    'convnext_base_in22k'

# VinDr-Mammo
eval_models = ['EfficientNet_b0', 'mobilenetv4_conv_small' ]  # bs: 24@L4  #, 'Resnet18'
#eval_models += ['densenet169', 'E2fficientNetV2-S_in21k']      # bs: 24@A100
#eval_models += ['EfficientNet_b3']                           # bs: 20@A100
#eval_models += ['convnext_base']        # bs: 14@A100
#eval_models += [''convnext_base_22k']        # bs: 14@A100


# CBIS-DDSM
#eval_models = ['Resnet18', 'Resnet50', 'mobilenet_v2'] #   bs: 8@T4
#eval_models += ['EfficientNet_b1'] #, 'EfficientNet_b2'
#eval_models += ['densenet121', 'mnasnet1_0'] #, 'resnext50_32x4d'
#eval_models += ['Resnet101', 'densenet169', 'EfficientNet-b3']

# from here on use bs=4
#eval_models += ['densenet201', 'mobilenet_v3_large', 'EfficientNet-b4']  # patch bs = 96|4

# Big ones
#eval_models += ['efficientnet_v2_s', 'convnext_tiny']  # patch bs = 96|4
#eval_models += ['E2fficientNetV2-M', 'convnext_small', 'convnext_base']  # patch bs = 96|4

# Models with 22k (ImageNet)
#eval_models += ['E2fficientNetV2-S_in21k', 'convnext_base_22k']  # patch bs = 96|4



# Single test
#eval_models = ['convnext_tiny']


In [7]:
#  Defines dataset and code files to Download

if DATASET == 'CBIS-DDSM':
    if INPUT_SIZE == '2304x1792':         # Select image size = only for CBIS
        prefix = 'data_cbis_v3_resized_2304x1792'
        dataset_file_full = 'data_cbis_v3_resized_2304x1792.tar.bz2'
    elif INPUT_SIZE == '1152x896':
        #prefix = 'data_resize_cbis_v2'
        prefix = 'cbis_ddsm_rev'                    # versão revisada e corrigida
        #dataset_file_full = 'cbis-ddsm-v2.tar.bz2'
        dataset_file_full = 'cbis_ddsm_rev.tar.bz2' # versão revisada e corrigida
        dataset_remote_folder = '/content/drive/My Drive/datasets/CBIS-DDSM/'
        if not COLLAB:
            #prefix = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2'
            prefix = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2_rev'
elif DATASET == 'VINDR_MAMMO-SMALL':
    prefix = 'data_resize_vindr_4sigma_small'
    dataset_remote_folder = '/content/drive/My Drive/datasets/VINDR-MAMMO/'
    dataset_file_full = 'data_resize_vindr_4sigma_small.tar.bz2'
    if not COLLAB:
        prefix = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/vindr-mammo/data_resize_vindr_4sigma_small'
elif DATASET == 'VINDR_MAMMO':
    prefix = 'data_resize_4sigma'
    dataset_remote_folder = '/content/drive/My Drive/datasets/VINDR-MAMMO/'
    # dataset_file_full = 'data_resize_vindr_4sigma.tar.bz2'
    dataset_file_full = 'data_resize_vindr_4sigma.tar'
    if not COLLAB:
        #prefix = '/media/dpetrini/DGPP02_1TB/usp/sinai_data/vindr-mammo/data_resize_4sigma'
        #prefix = '/media/dpetrini/PETRINI/datasets/vindr-mammo/data_resize_4sigma'
        prefix = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/vindr-mammo/'


if DATASET in 'VINDR_MAMMO' and INPUT_SIZE == '2304x1792':
    print('Wrong selection of input size...')

######################### New in 2026

prefix_test = prefix    # If not test dataset, test prefix is from same only dataset, otherwise will be changed below

if NET_TYPE == 'ONLY_TEST' and ONLY_TEST_DATASET == 'VINDR_MAMMO':
    prefix_test = 'data_resize_4sigma'
    # dataset_remote_folder_test = '/content/drive/My Drive/datasets/VINDR-MAMMO/'
    # dataset_file_full_test = 'data_resize_vindr_4sigma.tar'
    dataset_remote_folder_test = '/content/drive/My Drive/datasets/VINDR-MAMMO/'
    dataset_file_full_test = 'data_resize_vindr_4sigma_small.tar.bz2'
    if not COLLAB:
        # prefix_test = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/vindr-mammo/data_resize_4sigma'
        prefix_test = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/vindr-mammo/data_resize_vindr_4sigma_small'
        prefix = prefix_test  # can be the same justo to fullfil dataloader


# code files
code_remote_folder = '/content/drive/My Drive/sinai/scripts_final/'
code_file = 'scripts_patch_single_clf.tar.bz2'

In [8]:
# Defines exclusive for this experiment (single)

VAL_RESULT_FILE = save_path+project_name+ '_'+INPUT_SIZE+'_'+NET_TYPE+'_train_result.json'
TEST_RESULT_FILE = save_path+project_name+ '_'+INPUT_SIZE+'_'+NET_TYPE+'_test_result.json'

In [9]:
if COLLAB:
    import os
    from shutil import copyfile
    import subprocess
    from collab_support import download_expand_tar, finish_and_leave

    # Change to default collab path
    if os.getcwd() != ('/content'):
        os.chdir('/content')

    print('List of files in dataset remote dir: ')
    for file in os.listdir(dataset_remote_folder):
        print(file)

    # create project folder in /content - we are here - only for collab
    if os.path.isdir(save_path) is False:
        os.makedirs(save_path)
        print(f'Creating dir {save_path}')
    else:
        print(f'Save path: {save_path} already created')

    if os.path.isdir(project_name) is False:
        os.makedirs(project_name)

    os.chdir(project_name)
    print('Local project folder: ', os.getcwd())

In [10]:
# Copy remote datasets & code
if COLLAB:
    print('Processing Full images: ', os.path.join(dataset_remote_folder, dataset_file_full))
    download_expand_tar(dataset_remote_folder, dataset_file_full)
    print('Code files: ', os.path.join(code_remote_folder, code_file))
    download_expand_tar(code_remote_folder, code_file)
    if ONLY_TEST_DATASET == 'VINDR_MAMMO':
        print('Processing [test] Full images: ', os.path.join(dataset_remote_folder_test, dataset_file_full_test))
        download_expand_tar(dataset_remote_folder_test, dataset_file_full_test)

In [11]:
import sys
# Download 'nova' package that contais Trainer
if COLLAB:
    !rm -rf nova
    !git clone http://www.github.com/dpetrini/nova

    sys.path.insert(1, '/content/'+ project_name + '/nova')

    !pip -q install wandb

# nova trainer object
from trainer import Trainer

In [12]:
from collections import defaultdict
import os
import time
import gc
import sys
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

if COLLAB:
    !pip3 -q install timm

from dataset_full import MyDataset   # load our dataset class
from utils.model_utils import load_model
from utils.get_max_bsize_gpu import find_max_bs
from utils.general import print_finish_time, merge_two_dicts, \
                          get_samples_n, print_layers_names, count_model_parameters, \
                          get_best_val, get_test_mean, get_instance


device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

/home/dpetrini/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Will try to convert  158  images.
........................................Converted  40  images.
Dispositivo utilizado: cuda


In [13]:
######################## START PATCH

In [14]:
# preparações especificas
if NET_TYPE == 'PATCH_BASED':
    prefix_patch = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/'
    # JSON files to store training and test results
    PATCH_VAL_RESULT_FILE = save_path+project_name +'_patch_train_result.json'
    PATCH_TEST_RESULT_FILE = save_path+project_name +'_patch_test_result.json'

    # Find max batch size for each network for current GPU (single GPU only)
    max_bs_patch = find_max_bs(eval_models, input_size=(3, 224, 224))       # patch size
    print(f'Patch size: (3, 224, 224), batch: {max_bs_patch}')
    gc.collect()
    torch.cuda.empty_cache()

    TINY = '_tiny'           # '' or '_tiny'    to load test dataset for code testing

    # Defines for single classifier parameters (top layer)
    TOP_MODE = 'TOP_EF_NET'   #'TOP_RESNET'
    TOP_LAYER_N_BLOCKS = 2
    STRIDES = 2
    PRE_TRAINED_PATCH = True

    # Find path for patches
    if DATASET == 'CBIS-DDSM':
        prefix_patch = prefix_patch+'data_patches_S10_v2'+TINY
        NUM_CLASSES = 5
    elif DATASET == 'VINDR_BIRADS':
        prefix_patch = prefix_patch+'data-vindr-patch-sup-birads'+TINY
        NUM_CLASSES = 4

In [15]:
# Patch dataloader configuration

if NET_TYPE == 'PATCH_BASED':
    train_image_paths = prefix_patch+'/train'
    if DATASET == 'CBIS-DDSM':
        val_image_paths = prefix_patch+'/validation'
    elif DATASET == 'VINDR_BIRADS':
        val_image_paths = prefix_patch+'/val'
    test_image_paths = prefix_patch+'/test'

    # classe dataset que carregar arquivos e faz transformacoes
    dataset_train = MyDataset(train_image_paths, set_type='train')
    dataset_val = MyDataset(val_image_paths, set_type='val')
    dataset_test = MyDataset(test_image_paths, set_type='test')

    print('Size train:', len(dataset_train), ' Size val: ', len(dataset_val))

    patch_val_results = defaultdict(list)
    patch_test_results = defaultdict(list)

    # Opening JSON file
    if os.path.isfile(PATCH_VAL_RESULT_FILE):
        f = open(PATCH_VAL_RESULT_FILE)
        patch_val_results = merge_two_dicts(patch_val_results, json.load(f))
    if os.path.isfile(PATCH_TEST_RESULT_FILE):
        f = open(PATCH_TEST_RESULT_FILE)
        patch_test_results = merge_two_dicts(patch_test_results, json.load(f))

In [16]:
# PATCH CLF PART ###########################################
if NET_TYPE == 'PATCH_BASED':
    print('\nStarting Patch Clf ...\n')

    for j, model_name in enumerate(eval_models):

        if model_name in patch_val_results.keys():
            assert len(patch_val_results[model_name]) == number_runs, 'Runs number and length doesnt match.'
            print(f'Already run: {model_name}, runs: {len(patch_val_results[model_name])},', end='')
            val_acc = 0
            for k in range(len(patch_val_results[model_name])):
                val_acc += patch_val_results[model_name][k]['best_metric']['Accuracy']
            print(f' Val. Accuracy mean: {float(val_acc/number_runs):1.4}, Best Test: ', end='')
            print(f'{patch_test_results[model_name][8]['best_test']:1.4f}')   #  Esse 8 eh o indice da do defaultdict(list)
            continue


        # Carregando dataloaders aqui para usar BS dinamico, para cada rede
        dyn_batch_size = max_bs_patch[model_name]
        print(f'\nUsing dynamic bs: {dyn_batch_size}')
        train_dataloader = DataLoader(dataset_train, batch_size=dyn_batch_size,
                                    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        val_dataloader = DataLoader(dataset_val, batch_size=dyn_batch_size,
                                    shuffle=False, num_workers=NUM_WORKERS+1, pin_memory=True)
        test_dataloader = DataLoader(dataset_test, batch_size=dyn_batch_size,
                                    shuffle=False, num_workers=1, pin_memory=True)

        start_time = time.perf_counter()

        for i in range(number_runs):
            print(f'\n*************************************\nModel: {model_name}, run: {i+1}\n')
            model = load_model('patch', model_name, None, weights=PRE_TRAINED_PATCH,
                                patch_weights=False,   # maybe useless, pls check
                                num_classes=NUM_CLASSES)
            model = model.to(device)
            loss_func = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=LR)
            optim_args = {}

            # Below line to dump net (define param_stages)- never delete
            if PRINT_LAYER_NAMES:
                print_layers_names(model)

            train_config = {
                'num_epochs': NUM_EPOCHS,
                'batch_size': dyn_batch_size,  #MINI_BATCH_PATCH,
                'name': 'patch_' + DATASET + '_' + model_name.lower(),
                'title': DATASET + ' ' + project_name.upper() + ' Patch clf '+model_name,
                'stable_metric': 10,     # extend epochs number to wait N epochs with no metric change
                'save_best': True,
                'show_plots': False,
                'make_plots': True,
                'features': [],         # ['LR_SchedCB_W_Cos'],  # lr_warmup_cyc_cos LR_SchedCB_W_Cos 'lr_step_patch'
                'save_path': save_path,
            }

            session = Trainer(model, train_dataloader, val_dataloader, loss_func,
                            optimizer, optim_args, device, train_config)

            # train the model
            patch_val_results[model_name].append(session.train_and_validate_amp(return_dict=True))

            print("\nRunning models in test set... ", end='')
            patch_test_results[model_name].append(session.run_test(test_dataloader, "best"))

        elapsed_time = time.perf_counter() - start_time
        n_parameters = count_model_parameters(model)

        # cleanup dataloaders
        del train_dataloader, val_dataloader, test_dataloader
        gc.collect()

        # Calculate mean of test runs
        main, std = [], []
        val_acc, high_val_acc = 0, 0
        best_model = ''
        for i in range(number_runs):
            # get run result for mean
            content = patch_test_results[model_name][i]
            print('content', i, content)
            main.append(content)

            # Clean tensor information from training
            for name in patch_val_results[model_name][i]['metric_name']:
                metric_name = name
                patch_val_results[model_name][i]['best_metric'][metric_name] = float(patch_val_results[model_name][i]['best_metric'][metric_name].cpu().detach().numpy())

            # We only want metric_name = 'Accuracy' here (patch clf)
            cur_val = patch_val_results[model_name][i]['best_metric']['Accuracy']
            print(i, 'val_acc ', cur_val)
            if  cur_val > val_acc:
                val_acc = cur_val
                high_val_acc = i

        # handle average
        main = torch.tensor(main)
        std = torch.std(main)
        test_mean = f'{torch.mean(main):1.4f}±{torch.mean(std):1.4f}'
        print(f'{model_name} Test Avg: {test_mean}')

        # handle best model
        best_model = patch_val_results[model_name][high_val_acc]['best_model_file']['Accuracy']
        print('index', high_val_acc, 'best_model', best_model)

        # Get model selection, best val and correspondent test
        best_val = patch_val_results[model_name][high_val_acc]['best_metric']['Accuracy']
        best_val_epoch = patch_val_results[model_name][high_val_acc]['best_metric_epoch']['Accuracy']
        best_val_loss = patch_val_results[model_name][high_val_acc]['loss_plot']['Accuracy']
        best_val_acc_graph = patch_val_results[model_name][high_val_acc]['metric_plot']['Accuracy']
        best_val_test = patch_test_results[model_name][high_val_acc]

        # put mean and best model location in test_results
        patch_test_results[model_name].append({'best_val_index': high_val_acc})       # from nruns, which run is best
        patch_test_results[model_name].append({'best_val': best_val})                 # best val from best run
        patch_test_results[model_name].append({'best_epoch': best_val_epoch})         # epoch from best val from best run
        patch_test_results[model_name].append({'batch_size': dyn_batch_size})         # batch size from training
        patch_test_results[model_name].append({'total_train_time': int(elapsed_time)}) # Time in seconds spent in all runs during training
        patch_test_results[model_name].append({'n_parameters': n_parameters}) 
        patch_test_results[model_name].append({'best_test': best_val_test})           # result in test set from best run (model selected from best val out of nruns)
        patch_test_results[model_name].append({'test_mean': test_mean})               # test set mean in nruns
        patch_test_results[model_name].append({'best_loss': best_val_loss})           # loss graph from best run training process
        patch_test_results[model_name].append({'best_acc_graph': best_val_acc_graph}) # best acc graph from best run trainin process
        patch_test_results[model_name].append({'best_model': best_model})             # model selected from highest val in best run (to be used in single training)

        # After NRUNS training is ready save in file storage
        with open(PATCH_VAL_RESULT_FILE, 'w') as fp:
            json.dump(patch_val_results, fp)
        with open(PATCH_TEST_RESULT_FILE, 'w') as fp:
            json.dump(patch_test_results, fp)

    # Clear GPU
    gc.collect()
    torch.cuda.empty_cache()


In [17]:
######################## END PATCH

In [18]:
# Find max batch size for each network for current GPU (single GPU only)
input_size = (3, int(INPUT_SIZE.split('x')[0]), int(INPUT_SIZE.split('x')[1]))
max_bs_single = find_max_bs(eval_models, input_size=input_size)     # full img size
print(f'Full img {input_size}: {max_bs_single}')

# Clear GPU
gc.collect()
torch.cuda.empty_cache()

Usando GPU: NVIDIA GeForce RTX 2060 SUPER
Modo de batch: power2

== Testando EfficientNet_b0 ==
Batch máximo (power2) para EfficientNet_b0: 4

== Testando mobilenetv4_conv_small ==
Batch máximo (power2) para mobilenetv4_conv_small: 16

Full img (3, 1152, 896): {'EfficientNet_b0': 4, 'mobilenetv4_conv_small': 16}


In [19]:
# Precisou disso para rodar alguns modelos - libera algumas centenas de MB na GPU
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Get the value of the environment variable
value = os.environ.get('PYTORCH_CUDA_ALLOC_CONF')

# Check if the environment variable exists
if value is not None:
    print(f"The value of ENV_VARIABLE_NAME is: {value}")
else:
    print("ENV_VARIABLE_NAME is not set.")


The value of ENV_VARIABLE_NAME is: expandable_segments:True


In [20]:
# MAIN
train_image_paths = prefix+'/train'
validation_image_paths = prefix+'/val'
test_image_paths = prefix_test+'/test'

# classe dataset que carregar arquivos e faz transformacoes
dataset_train = MyDataset(train_image_paths, set_type='train', dataset=DATASET)
dataset_val = MyDataset(validation_image_paths, set_type='val', dataset=DATASET)

dataset_test = MyDataset(test_image_paths, set_type='test_analysis', dataset=DATASET if ONLY_TEST_DATASET is None else ONLY_TEST_DATASET)

print('Size train:', len(dataset_train), ' Size val: ', len(dataset_val))

n_samples = len(dataset_train) + len(dataset_val)

single_val_results = defaultdict(list)
single_test_results = defaultdict(list)

# Opening JSON file
if os.path.isfile(VAL_RESULT_FILE):
    f = open(VAL_RESULT_FILE)
    single_val_results = merge_two_dicts(single_val_results, json.load(f))
if os.path.isfile(TEST_RESULT_FILE):
    f = open(TEST_RESULT_FILE)
    single_test_results = merge_two_dicts(single_test_results, json.load(f))

['benign', 'malign'] {'benign': 0, 'malign': 1}
[DataLoader] Loaded images samples size:  2234 /media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2_rev/train
Share: positive (malign): 1001, negative (benign): 1233
['benign', 'malign'] {'benign': 0, 'malign': 1}
[DataLoader] Loaded images samples size:  245 /media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2_rev/val
Share: positive (malign): 114, negative (benign): 131
['benign', 'malign'] {'benign': 0, 'malign': 1}
[DataLoader] Loaded images samples size:  595 /media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2_rev/test
Share: positive (malign): 246, negative (benign): 349
Size train: 2234  Size val:  245


In [21]:
# SINGLE IMAGE CLASSIFIER ################################################
print('\nStarting SINGLE image Clf ...\n')

print('Now selected: ', DATASET, INPUT_SIZE, NET_TYPE, number_runs)

for j, model_name in enumerate(eval_models):

    if model_name in single_val_results.keys():
        assert len(single_val_results[model_name]) == number_runs, 'Runs number and length doesnt match.'
        print(f'Already run: {model_name}, runs: {number_runs},', end='')
        val_acc = 0
        for k in range(len(single_val_results[model_name])):
            val_acc += single_val_results[model_name][k]['best_metric']['AUC']
        print(f' Val. AUC mean: {float(val_acc/number_runs):1.4}, Best Test: ', end='')
        best_test_res = get_instance(single_test_results[model_name], 'test_result')
        print(best_test_res)
        continue

    # DataLoader eh iterador que retorna tensores pytorch
    dyn_batch_size = max_bs_single[model_name]
    print(f'\nUsing dynamic bs: {dyn_batch_size}')
    train_dataloader = DataLoader(dataset_train, batch_size=dyn_batch_size,
                                shuffle=True,
                                num_workers=2, pin_memory=True,
                                drop_last=True)   # drop_last for ResNest50...
    val_dataloader = DataLoader(dataset_val, batch_size=dyn_batch_size,
                                shuffle=False, num_workers=2, pin_memory=True,
                                drop_last=True)
    test_dataloader = DataLoader(dataset_test, batch_size=1, # easier for results storage
                                shuffle=False, num_workers=1, pin_memory=True,
                                drop_last=True)

    start_time = time.perf_counter()
    
    for i in range(number_runs):

        print(f'\n*************************************\nModel: {model_name}, run: {i+1}')

        #device = torch.device('cuda:0')
        if NET_TYPE == 'LEARN2RESIZE':
          model = load_model('learn_resize_input', model_name, None, weights=True,
                             num_classes=2)
        elif NET_TYPE == 'SINGLE_PURE':
          model, inplanes, extract_layers = load_model('single_pure', model_name, None, weights=True,
                                        num_classes=2)
        elif NET_TYPE == 'RANDOM_WEIGHTS':
          model, inplanes, extract_layers = load_model('single_pure', model_name, None, weights=False,
                             num_classes=2)
        elif NET_TYPE == 'FIXED_RESIZE':
          model = load_model('fixed_resize_input', model_name, None, weights=True,
                             num_classes=2)
          
        elif NET_TYPE == 'PATCH_BASED':       # Suport patch based classifiers, patch trained above somewhere

            # Retrieve best patch model from list
            patch_model_file = None
            for item in patch_test_results[model_name]:
                if isinstance(item, dict):
                    if 'best_model' in item:
                        patch_model_file = item['best_model']
            print('Best [patch] Model: ', patch_model_file)

            model, inplanes, extract_layers  = load_model('single', model_name, patch_model_file,
                                                weights=False,patch_weights=True,
                                                top_mode=TOP_MODE, top_layer_blocks=TOP_LAYER_N_BLOCKS,
                                                strides=STRIDES, num_classes=NUM_CLASSES)

        elif NET_TYPE in ['TRANSFER', 'ONLY_TEST']:
            # Let's load the transfer / test-only model, likely trained in different dataset
            original_test_results = defaultdict(list)

            # Opening JSON file from pre-trained dataset
            if os.path.isfile(ORIGINAL_RESULT_FILE):
                f = open(ORIGINAL_RESULT_FILE)
                original_test_results = merge_two_dicts(original_test_results, json.load(f))

            # Retrieve best model from json
            cross_model_file = None
            for item in original_test_results[model_name]:
                if isinstance(item, dict):
                    if 'best_model' in item:
                        cross_model_file = item['best_model']
            if cross_model_file is None:
               print(f'Best [{model_name}] Model not found.\n')
               sys.exit()
            print(f'Best [{model_name}] Model loaded for {'transfer learning' if NET_TYPE == 'TRANSFER' else 'test only'}: {cross_model_file}')

            # Finally load model
            model = load_model('transfer', model_name, cross_model_file, weights=False,
                               num_classes=2)

        else:
          print('Wrong selection of NET_TYPE')
          sys.exit()

        model = model.to(device)            # load_model nao deve carregar para device, apenas aqui

        # Loss function
        loss_func = nn.CrossEntropyLoss()

        # Optimize
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        optim_args = {'base_lr': LR, 'delta': 2e-5, 'period': NUM_EPOCHS//10, 'warmup': 4}

        # Below line to dump net (define param_stages)- never delete
        if PRINT_LAYER_NAMES:
            print_layers_names(model)

        train_config = {
            'num_epochs': NUM_EPOCHS,
            'batch_size': dyn_batch_size,
            'name': 'single_' + DATASET + '_' + model_name.lower(),
            'title': project_name.upper() + ' Single clf ' \
                     +model_name[0].upper()+model_name[1:].lower() \
                     + ' ' + DATASET,
            'save_last': False,
            'save_best': True,
            'stable_metric': 10,
            'features': ['auc', 'lr_warmup_cyc_cos'],
            'make_plots': True,
            'show_plots': False,
            'use_wandb': False,
            'save_path': save_path,
        }

        session = Trainer(model, train_dataloader, val_dataloader, loss_func,
                          optimizer, optim_args, device, train_config)

        # train the model - Normal Path
        if NET_TYPE != 'ONLY_TEST':
            single_val_results[model_name].append(session.train_and_validate_amp(return_dict=True))
            m, n = get_samples_n(DATASET)
            print("\nRunning models in test set...")
            single_test_results[model_name].append(session.run_test_auc(test_dataloader,
                                                                    "best", m=m, n=n,
                                                                    ONLY_TEST_DATASET=ONLY_TEST_DATASET))

        else:       # TEST ONLY Path
            m, n = get_samples_n(ONLY_TEST_DATASET)
            print("\nRunning models in test set (only)...")
            single_test_results[model_name].append(session.run_test_only_auc(test_dataloader,
                                                                             m, n,
                                                                             ONLY_TEST_DATASET))


    elapsed_time = time.perf_counter() - start_time
    n_parameters = count_model_parameters(model)

    # cleanup dataloaders
    del train_dataloader, val_dataloader, test_dataloader
    gc.collect()
    
    high_val_auc = best_val = best_val_epoch = best_val_loss = 0
    best_val_acc_graph = best_val_test = best_val_test_plot = best_model = 0

    if NET_TYPE != 'ONLY_TEST':         # (training related info)
        # [Model selection] Get highest validation model index (high_val_auc) #############
        high_val_auc = get_best_val(number_runs, single_val_results, model_name)
        max_auc = single_val_results[model_name][high_val_auc]['best_metric']['AUC']
        print(f'{model_name} HIGHER INDEX : {high_val_auc} higher AUC: {max_auc}')

        # Model selection: best model from validation
        best_model = single_val_results[model_name][high_val_auc]['best_model_file']['AUC']
        print('index', high_val_auc, 'best_model', best_model)

        # Retrieve validation results from best run, to store in final results
        best_val = single_val_results[model_name][high_val_auc]['best_metric']['AUC']
        best_val_epoch = single_val_results[model_name][high_val_auc]['best_metric_epoch']['AUC']
        best_val_loss = single_val_results[model_name][high_val_auc]['loss_plot']['Accuracy']     # Loss soh grava via acc.
        best_val_acc_graph = single_val_results[model_name][high_val_auc]['metric_plot']['AUC']

        # Test Result from the best validation model (not be best test model!)
        best_val_test = single_test_results[model_name][high_val_auc][0]  # 2025
        best_val_test_plot = single_test_results[model_name][high_val_auc][1] # 2025

    # Calculate mean of test runs
    test_mean = get_test_mean(number_runs, single_test_results, model_name)
    print(f'{model_name} Test Avg: {test_mean}')

    # Some conditional insertions in result file
    if ONLY_TEST_DATASET is not None and NET_TYPE == 'ONLY_TEST':
        data_text = f'Test only: {ONLY_TEST_DATASET}'
        single_test_results[model_name].append({'dataset_info': data_text})
        single_test_results[model_name].append({'tested_from': cross_model_file}) # ORIGINAL_RESULT_FILE
    else:
        data_text = f'Train and test: {DATASET}'    # Do nothing
    
    if NET_TYPE == 'TRANSFER':           
        single_test_results[model_name].append({'transfered_from': cross_model_file}) # ORIGINAL_RESULT_FILE

    # # General insertions in test_result_file JSON [Validation]
    single_test_results[model_name].append({'best_val_index': high_val_auc})
    single_test_results[model_name].append({'best_val': best_val})
    single_test_results[model_name].append({'best_epoch': best_val_epoch})
    single_test_results[model_name].append({'batch_size': dyn_batch_size})
    single_test_results[model_name].append({'best_loss': best_val_loss})
    single_test_results[model_name].append({'best_acc_graph': best_val_acc_graph})
    
    # # General insertions in test_result_file JSON [Test]
    single_test_results[model_name].append({'total_train_time': int(elapsed_time)}) # Time in seconds spent in all runs during training
    single_test_results[model_name].append({'n_parameters': n_parameters})          # number of parameters of this model, in millions
    single_test_results[model_name].append({'test_result': best_val_test})          # Test Result from the best validation model
    single_test_results[model_name].append({'test_mean': test_mean})
    single_test_results[model_name].append({'test_result_plot': best_val_test_plot})
    single_test_results[model_name].append({'best_model': best_model})

    if NET_TYPE in ['SINGLE_PURE', 'PATCH_BASED']:       #, 'RANDOM_WEIGHTS']
        single_test_results[model_name].append({'inplanes': inplanes})
        single_test_results[model_name].append({'extract_layers': extract_layers})

    # After NRUNS training is ready save in file storage
    with open(VAL_RESULT_FILE, 'w') as fp:
        json.dump(single_val_results, fp)
    with open(TEST_RESULT_FILE, 'w') as fp:
        json.dump(single_test_results, fp)

# Clear GPU
gc.collect()
torch.cuda.empty_cache()


Starting SINGLE image Clf ...

Now selected:  CBIS-DDSM 1152x896 SINGLE_PURE 2

Using dynamic bs: 4

*************************************
Model: EfficientNet_b0, run: 1
Loading [single_pure] model: EfficientNet_b0 file: None for Single image clf
DC_SINGLE_PURE Single clf Efficientnet_b0 CBIS-DDSM
Creating dir ./dc_single_pure_results_CBIS-DDSM_2026-06-29-20h12m/models_single_CBIS-DDSM_efficientnet_b0
Creating dir ./dc_single_pure_results_CBIS-DDSM_2026-06-29-20h12m/plots_single_CBIS-DDSM_efficientnet_b0
Warm-up + Cyclic Cosine Learning Rate scheduler

Registered Callbacks:  BaseCB  AUC_CB  LR_SchedCB_W_Cyc_Cos  
BaseCB  AUC_CB  LR_SchedCB_W_Cyc_Cos  Using AMP

Epoch: 1/2
Updating 213 layers current learning rate is: 2.50e-05
▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒░░░░░░░░░░ |------>  Best Val Acc model now 0.6230
Epoch: 001, Train: Loss: 0.6688, Acc: 57.57%, Val: Loss: 0.6603, Acc: 62.30%, Time: 147.64s
[Train] AUC Malignant: 0.609 [Val] AUC Val Malignant: 0.711 |------>  

In [22]:
###################### TWO-Views Classifier code

In [23]:
from multiple.dataset_CBIS_DDSM_RAM import MyDataset   # load our dataset class
from multiple.two_views_net import SideMIDBreastModel, SideTOPBreastModel

TOPOLOGY = 'side_mid_clf'     # 'side_mid_clf' 'side_top_clf' 'side_function_clf' 'side_fc_clf'  'side_mid_thin_clf' 'side_mid_clf_nopool'

TARGET = TOPOLOGY.lower() + '_' + NET_TYPE.lower() 
TOP_LAYER_N_BLOCKS = 2
STRIDES = 2
TOP_LAYER_BLOCK_TYPE = 'mbconv'   # 'mbconv'  'resnet'
USE_AVG_POOL = True

# MINI_BATCH = 2      #  dyn_batch_size/2

In [24]:

# Source of images for this session (full images divided in benign/malign)
if DATASET == 'CBIS-DDSM':
    prefix = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2'
elif DATASET == 'VINDR_MAMMO-SMALL':
    prefix = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/vindr-mammo/data_resize_4sigma_small'
elif DATASET == 'VINDR_MAMMO':
    # prefix = '/media/dpetrini/DGPP02_1TB1/usp/sinai_data/vindr-mammo/data_resize_4sigma'
    prefix = '/home/dpetrini/devel/apagar/data_resize_4sigma'
else:
    print('Please check dataset')
    sys.exit()


# classe dataset que carregar arquivos e faz transformacoes
dataset_train = MyDataset(prefix+'/train', set_type='train', dataset=DATASET)
dataset_val = MyDataset(prefix+'/val', set_type='val', dataset=DATASET)
dataset_test = MyDataset(prefix+'/test', set_type='test_analysis', dataset=DATASET)

# # DataLoader eh iterador que retorna tensores pytorch
# train_dataloader = DataLoader(dataset_train, batch_size=int(MINI_BATCH),
#                             shuffle=True,
#                             num_workers=2, pin_memory=True)
# val_dataloader = DataLoader(dataset_val, batch_size=int(MINI_BATCH),
#                             shuffle=False, num_workers=2, pin_memory=True)
# test_dataloader = DataLoader(dataset_test, batch_size=1,
#                             shuffle=False, num_workers=1, pin_memory=True,
#                             drop_last=True)

Accounting for list size differences
Total: 963 pairs [CC, MLO], positive (malign): 434, negative (benign): 529
[DataLoader] Read images train size: 963 pairs /media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2/train
[DataLoader] Loading train images on RAM...
Reached max ram size
[DataLoader] Loaded 10 pairs of images (CC, MLO): 123863040 bytes.
Accounting for list size differences
Total: 111 pairs [CC, MLO], positive (malign): 50, negative (benign): 61
[DataLoader] Read images val size: 111 pairs /media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2/val
Accounting for list size differences
Total: 273 pairs [CC, MLO], positive (malign): 111, negative (benign): 162
[DataLoader] Read images test_analysis size: 273 pairs /media/dpetrini/DGPP02_1TB1/usp/sinai_data/cbis-ddsm-v2/data_resize_cbis_v2/test


In [ ]:
# Two-Views Classifier training

TWO_VIEWS_VAL_RESULT_FILE  = save_path + project_name + '_' + INPUT_SIZE + '_' + NET_TYPE + '_two_views_train_result.json'
TWO_VIEWS_TEST_RESULT_FILE = save_path + project_name + '_' + INPUT_SIZE + '_' + NET_TYPE + '_two_views_test_result.json'

two_views_val_results  = defaultdict(list)
two_views_test_results = defaultdict(list)

if os.path.isfile(TWO_VIEWS_VAL_RESULT_FILE):
    with open(TWO_VIEWS_VAL_RESULT_FILE) as f:
        two_views_val_results = merge_two_dicts(two_views_val_results, json.load(f))
if os.path.isfile(TWO_VIEWS_TEST_RESULT_FILE):
    with open(TWO_VIEWS_TEST_RESULT_FILE) as f:
        two_views_test_results = merge_two_dicts(two_views_test_results, json.load(f))

print('\nStarting Two-views image Clf ...\n')
print('Now selected: ', DATASET, INPUT_SIZE, NET_TYPE, number_runs)

for j, model_name in enumerate(eval_models):

    if model_name in two_views_val_results.keys():
        assert len(two_views_val_results[model_name]) == number_runs, 'Runs number and length doesnt match.'
        print(f'Already run: {model_name}, runs: {number_runs},', end='')
        val_acc = 0
        for k in range(len(two_views_val_results[model_name])):
            val_acc += two_views_val_results[model_name][k]['best_metric']['AUC']
        print(f' Val. AUC mean: {float(val_acc/number_runs):1.4}, Best Test: ', end='')
        best_test_res = get_instance(two_views_test_results[model_name], 'test_result')
        print(best_test_res)
        continue

    # Retrieve information from single model training
    full_model_file = None
    for item in single_test_results[model_name]:
        if isinstance(item, dict):
            if 'best_model' in item:
                full_model_file = item['best_model']
            if 'inplanes' in item:
                inplanes = item['inplanes']
            if 'extract_layers' in item:
                extract_layers = item['extract_layers']
    print('Best [full image] Model: ', full_model_file, inplanes, extract_layers)

    dyn_batch_size = max(1, int(max_bs_single[model_name] / 2))
    print(f'\nUsing dynamic bs: {dyn_batch_size}')

    # DataLoader eh iterador que retorna tensores pytorch
    train_dataloader = DataLoader(dataset_train, batch_size=dyn_batch_size,
                                shuffle=True, num_workers=2, pin_memory=True)
    val_dataloader = DataLoader(dataset_val, batch_size=dyn_batch_size,
                                shuffle=False, num_workers=2, pin_memory=True)
    test_dataloader = DataLoader(dataset_test, batch_size=1, 
                                shuffle=False, num_workers=1, pin_memory=True,
                                drop_last=True)

    start_time = time.perf_counter()

    for i in range(number_runs):

        print(f'\n*************************************\nModel: {model_name}, run: {i+1}\n')

        if TOPOLOGY == 'side_top_clf':
            model = SideTOPBreastModel(device, full_model_file, NET_TYPE)
        elif TOPOLOGY == 'side_mid_clf':
            model = SideMIDBreastModel(device, full_model_file, model_name, TOP_LAYER_N_BLOCKS,
                                       b_type=TOP_LAYER_BLOCK_TYPE, avg_pool=USE_AVG_POOL,
                                       strides=STRIDES,
                                       exp_type=NET_TYPE,
                                       dataset=DATASET,
                                       inplanes=inplanes,
                                       extract_layers=extract_layers)

        model = model.to(device)

        optim_args = {'base_lr': LR, 'delta': 2e-5, 'period': NUM_EPOCHS//10, 'warmup': 4}

        loss_func = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        if PRINT_LAYER_NAMES:
        # if True:
            print_layers_names(model)
            print(model)

        train_config = {
            'num_epochs': NUM_EPOCHS,
            'batch_size': dyn_batch_size,
            'name': 'two-views_' + DATASET + '_' + model_name.lower(),
            'title': project_name.upper() + ' Two-views clf '
                      + model_name[0].upper() + model_name[1:].lower()
                      + ' ' + DATASET,
            'save_last': False,
            'save_best': True,
            'features': ['auc', 'lr_warmup_cyc_cos'],
            'show_plots': False,
            'make_plots': True,
            'stable_metric': 10,
            'use_wandb': False,
            'save_path': save_path,
        }

        session = Trainer(model, train_dataloader, val_dataloader, loss_func,
                          optimizer, optim_args, device, train_config)

        # train the model
        two_views_val_results[model_name].append(session.train_and_validate_amp(return_dict=True))

        # Test set evaluation
        m, n = get_samples_n(DATASET)
        print("\nRunning models in test set...")
        two_views_test_results[model_name].append(
            session.run_test_auc(test_dataloader, "best", m=m, n=n))

    elapsed_time = time.perf_counter() - start_time
    n_parameters = count_model_parameters(model)

    gc.collect()

    # Model selection: run with best validation AUC
    high_val_auc = get_best_val(number_runs, two_views_val_results, model_name)
    max_auc = two_views_val_results[model_name][high_val_auc]['best_metric']['AUC']
    print(f'{model_name} HIGHER INDEX : {high_val_auc} higher AUC: {max_auc}')

    best_model        = two_views_val_results[model_name][high_val_auc]['best_model_file']['AUC']
    best_val          = two_views_val_results[model_name][high_val_auc]['best_metric']['AUC']
    best_val_epoch    = two_views_val_results[model_name][high_val_auc]['best_metric_epoch']['AUC']
    best_val_loss     = two_views_val_results[model_name][high_val_auc]['loss_plot']['Accuracy']
    best_val_acc_graph= two_views_val_results[model_name][high_val_auc]['metric_plot']['AUC']
    best_val_test     = two_views_test_results[model_name][high_val_auc][0]
    best_val_test_plot= two_views_test_results[model_name][high_val_auc][1]
    print('index', high_val_auc, 'best_model', best_model)

    test_mean = get_test_mean(number_runs, two_views_test_results, model_name)
    print(f'{model_name} Test Avg: {test_mean}')

    # Metadata in test results (mirrors single_test_results pattern)
    two_views_test_results[model_name].append({'best_val_index':    high_val_auc})
    two_views_test_results[model_name].append({'best_val':          best_val})
    two_views_test_results[model_name].append({'best_epoch':        best_val_epoch})
    two_views_test_results[model_name].append({'batch_size':        dyn_batch_size})
    two_views_test_results[model_name].append({'best_loss':         best_val_loss})
    two_views_test_results[model_name].append({'best_acc_graph':    best_val_acc_graph})
    two_views_test_results[model_name].append({'total_train_time':  int(elapsed_time)})
    two_views_test_results[model_name].append({'n_parameters':      n_parameters})
    two_views_test_results[model_name].append({'test_result':       best_val_test})
    two_views_test_results[model_name].append({'test_mean':         test_mean})
    two_views_test_results[model_name].append({'test_result_plot':  best_val_test_plot})
    two_views_test_results[model_name].append({'best_model':        best_model})
    two_views_test_results[model_name].append({'topology':          TOPOLOGY})

    # Save to JSON files after each model
    with open(TWO_VIEWS_VAL_RESULT_FILE, 'w') as fp:
        json.dump(two_views_val_results, fp)
    with open(TWO_VIEWS_TEST_RESULT_FILE, 'w') as fp:
        json.dump(two_views_test_results, fp)


Starting Two-views image Clf ...

Now selected:  CBIS-DDSM 1152x896 SINGLE_PURE 2
Best [full image] Model:  ./dc_single_pure_results_CBIS-DDSM_2026-06-29-20h12m/models_single_CBIS-DDSM_efficientnet_b0/2026-06-29-20h20m_2ep_best_model_AUC_07164.pt 1280 2

Using dynamic bs: 2

*************************************
Model: EfficientNet_b0, run: 1

Loading [transfer] model: EfficientNet_b0 file: ./dc_single_pure_results_CBIS-DDSM_2026-06-29-20h12m/models_single_CBIS-DDSM_efficientnet_b0/2026-06-29-20h20m_2ep_best_model_AUC_07164.pt for Single image clf

Loading model for transfer learning.

1280 2
Creating Side Mid Networking using: EfficientNet_b0  and Top Block type:  mbconv  Qty:  2
Using EFBlocks (top block) parameters from EfficientNet-b0 [Two views creation].


NameError: name 'MINI_BATCH' is not defined

In [ ]:
###################### TWO-Views Classifier code

In [ ]:
# Finish everything
# cleanup dataloaders
del train_dataloader, val_dataloader, test_dataloader
gc.collect()
torch.cuda.empty_cache()
print(print_finish_time())
if COLLAB:
    # from google.colab import runtime
    # runtime.unassign()
    finish_and_leave()